This notebook is for cleaning my data

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df = pd.read_csv("data/raw_facilities.csv")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nShare missing per column:")
print(df.isna().mean().sort_values(ascending=False).round(3))

print("\nDuplicate rows:", df.duplicated().sum())

# Columns we care about: check each one exists before summarizing it
for col in ["latitude", "longitude"]:
    if col in df.columns:
        print(f"\n{col} (as numbers):")
        print(pd.to_numeric(df[col], errors="coerce").describe())
    else:
        print(f"\n!! '{col}' not found. Look for the right name in the Columns list above.")

for col in ["care_system", "mode of payment", "Subcounty"]:
    if col in df.columns:
        print(f"\n{col}: {df[col].nunique()} unique values")
        print(df[col].value_counts(dropna=False).head(15))
    else:
        print(f"\n!! '{col}' not found.")

for col in ["services", "operating_hours"]:
    if col in df.columns:
        print(f"\n{col} sample:")
        print(df[col].dropna().head(10).to_string())
    else:
        print(f"\n!! '{col}' not found.")

print("\nFirst 10 rows:")
display(df.head(10))

Shape: (6520, 11)

Columns: ['facility_name', 'services', 'latitude', 'longitude', 'rating', 'operating_hours', 'website', 'phone_number', 'care_system', 'mode of payment', 'Subcounty']

Share missing per column:
website            0.773
rating             0.331
phone_number       0.290
operating_hours    0.250
facility_name      0.000
services           0.000
latitude           0.000
longitude          0.000
care_system        0.000
mode of payment    0.000
Subcounty          0.000
dtype: float64

Duplicate rows: 51

latitude (as numbers):
count    6520.000000
mean        0.461870
std         0.890694
min       -26.094880
25%         0.304793
50%         0.323781
75%         0.388027
max        41.203322
Name: latitude, dtype: float64

longitude (as numbers):
count    6520.000000
mean       32.470321
std         1.506546
min       -77.194525
25%        32.547748
50%        32.584446
75%        32.611222
max        34.957535
Name: longitude, dtype: float64

care_system: 4 unique values

,facility_name,services,latitude,longitude,rating,operating_hours,website,phone_number,care_system,mode of payment,Subcounty
0,St. Francis Nyenga Hospital,"General Medical & Surgical Services, HIV/AIDS ...",0.379327,33.140027,3.5,"Monday: Open 24 hours,Tuesday: Open 24 hours,W...",NaN,NaN,PNFP,cash,Nyenga Division
1,Tongolo HC II,General Care,0.373296,33.100767,NaN,"N,/,A",NaN,0772 971109,GOVT,cash,Nyenga Division
2,Kasubi HC III,"Imaging Services, OPD, Specialized Clinics",0.327001,32.552565,NaN,"Monday: Open 24 hours,Tuesday: Open 24 hours,W...",NaN,0706 399074,GOVT,cash,Buikwe Subcounty
3,Buikwe HC III,"Maternal and Child Health, Outpatient, Inpatie...",0.344226,33.026341,NaN,"Monday: 8:00â€¯AMâ€‰â€“â€‰5:00â€¯PM,Tuesday: 8...",NaN,0782 014778,GOVT,cash,Buikwe Town Council
4,Buikwe St. Charles Lwanga Hospital,"General Medical & Surgical Services, HIV/AIDS ...",0.339290,33.032193,4.7,"Monday: Open 24 hours,Tuesday: Open 24 hours,W...",NaN,0774 342186,PNFP,cash,Buikwe Town Council
5,Health Initiative Association HC II,"Preventive Services, Healthcare Screenings, Ca...",0.373908,32.938132,NaN,"N,/,A",http://www.hiainternationals.com/,0783 610768,PFP,cash,Buikwe Town Council
6,Health Initiative For Africa HC III,"Maternal and Child Health, Family Planning, Re...",0.345100,33.028700,NaN,"N,/,A",NaN,NaN,PNFP,cash,Buikwe Town Council
7,Busagazi HC II,"Maternity, General Surgery, Pediatrics, Outpat...",0.603828,33.689332,5.0,"Monday: Closed,Tuesday: Open 24 hours,Wednesda...",NaN,NaN,PFP,cash,Najja Subcounty
8,Dema Care Clinic,"Facials, Botox, Dysport, facial fillers, skinc...",0.352712,32.615035,4.5,"Monday: 9:00â€¯AMâ€‰â€“â€‰6:00â€¯PM,Tuesday: 9...",https://www.dermaskinclinicug.com/,0782 932937,PFP,cash,Najja Subcounty
9,Kingdom Life Health Center Clinic,"Family Clinic, Rehabilitation Services, Emerge...",0.311491,32.553859,4.3,"Monday: Open 24 hours,Tuesday: Open 24 hours,W...",NaN,0706 303080,PFP,cash,Najja Subcounty


In [4]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw_facilities.csv")
audit = {"raw_rows": len(df)}

def fix_text(s):
    if not isinstance(s, str): return s
    try: return s.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError): return s

for col in df.select_dtypes(include=["object", "string"]):
    df[col] = df[col].map(fix_text)
    df[col] = df[col].str.replace("[\u202f\u2009]", " ", regex=True).str.strip()

is_na = df["operating_hours"].str.replace(",", "", regex=False).str.upper().eq("N/A")
audit["hours_NA_marker"] = int(is_na.sum())
df.loc[is_na, "operating_hours"] = np.nan

df = df.drop_duplicates(); audit["after_exact_dups"] = len(df)
df = df.drop_duplicates(subset=["facility_name", "latitude", "longitude"])
audit["after_name_loc_dups"] = len(df)

LAT, LON = (-1.5, 4.3), (29.5, 35.1)
ok = df.latitude.between(*LAT) & df.longitude.between(*LON)
audit["outside_uganda"] = int((~ok).sum())
print(df.loc[~ok, ["facility_name", "latitude", "longitude"]])
df = df[ok]
coord_key = df["latitude"].round(6).astype(str) + "," + df["longitude"].round(6).astype(str)
df["location_shared_by"] = coord_key.map(coord_key.value_counts()).astype(int)

centre = df.groupby("Subcounty")[["latitude", "longitude"]].transform("median")
p1, p2 = np.radians(df["latitude"]), np.radians(centre["latitude"])
a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(centre["longitude"] - df["longitude"]) / 2) ** 2
km_from_centre = 6371 * 2 * np.arcsin(np.sqrt(a))
enough_peers = df.groupby("Subcounty")["facility_name"].transform("size") >= 3
df["location_mismatch"] = enough_peers & (km_from_centre > 50)

audit["coordinates_shared_with_4plus_others"] = int((df["location_shared_by"] >= 5).sum())
audit["largest_shared_coordinate_group"] = int(df["location_shared_by"].max())
audit["far_from_own_subcounty"] = int(df["location_mismatch"].sum())

df["care_system_code"] = df["care_system"].str.upper()
df["care_system"] = df["care_system_code"].replace(
    {"GOVT": "Government", "PNFP": "Private not-for-profit", "PFP": "Private for-profit"})

audit["final_rows"] = len(df)
for c in ["phone_number", "operating_hours", "rating"]:
    audit[f"{c}_pct"] = round(float(df[c].notna().mean() * 100), 1)
print(audit)
df.to_csv("data/facilities.csv", index=False)

              facility_name   latitude  longitude
748   Hal-Pa General Clinic  41.203322 -77.194525
4545     Morningside Clinic -26.094880  28.055341
6372     Kafunjo-Nyakarambi  -1.593857  30.893133
{'raw_rows': 6520, 'hours_NA_marker': 539, 'after_exact_dups': 6469, 'after_name_loc_dups': 6412, 'outside_uganda': 3, 'coordinates_shared_with_4plus_others': 3605, 'largest_shared_coordinate_group': 728, 'far_from_own_subcounty': 1177, 'final_rows': 6409, 'phone_number_pct': 71.0, 'operating_hours_pct': 66.6, 'rating_pct': 67.0}
